## Excel_Challenge_930 - Start Stop RunTime

### Problem from Excel BI Challenge account on LinkedIn

### Solution author : M R Shaheedullah

In [1]:
# import libraries
import pandas as pd
import datetime

In [2]:
# load data
df = pd.read_csv("Excel_Challenge_930 - Start Stop RunTime.csv", parse_dates=['EventTime'])
df

,Machine,EventTime,EventType
0,M1,2024-01-03 08:00:00,Start
1,M1,2024-01-03 10:15:00,Stop
2,M1,2024-01-03 11:00:00,Start
3,M1,2024-01-03 13:30:00,Stop
4,M1,2024-01-03 15:00:00,Start
5,M2,2024-01-03 07:50:00,Start
6,M2,2024-01-03 09:10:00,Stop
7,M2,2024-01-03 10:30:00,Start
8,M2,2024-01-03 12:00:00,Stop
9,M2,2024-01-03 13:45:00,Start


In [3]:
# sort dataframe
df = df.sort_values(by=["Machine", "EventTime"], ascending = [True, True])
df = df.reset_index(drop=True)

In [4]:
all_machines=[]   # create list of machines 
all_machines = df['Machine'].unique().tolist()
# create a dfframe with the max dates for each machine 
all_max = df.groupby('Machine')['EventTime'].max().reset_index()
new_rows_list=[]  # create a list that is to hold new rows (as dicts)
stop_list = []  # create a list to hold index position where new ro is to be added

In [ ]:
for i, row in df.iterrows():
    for m in all_machines:
        # find max eventtime for this machine
        max_dt_df = all_max[all_max['Machine']==m]
        # return eventtime
        current_max = max_dt_df.loc[max_dt_df['Machine']==m, 'EventTime']
        # if the current row is equal to max for this machine and eventtype is 'Start'
        # a new row with eventtype 'Stop' needs to be added (with eventtime 18:00)
        if (row['EventTime']==current_max.iloc[0]) & (row['EventType'] == 'Start'):
        # current date as string
            current_date = pd.to_datetime(row['EventTime']).strftime('%Y-%m-%d')
            new_row = {'Machine': m, 'EventTime': current_date + " 18:00", 'EventType': 'Stop'}
            new_rows_list.append(new_row)
            stop_list.append(i)
            

In [6]:
df = pd.concat([df, pd.DataFrame(new_rows_list)], ignore_index=True)
df

,Machine,EventTime,EventType
0,M1,2024-01-03 08:00:00,Start
1,M1,2024-01-03 10:15:00,Stop
2,M1,2024-01-03 11:00:00,Start
3,M1,2024-01-03 13:30:00,Stop
4,M1,2024-01-03 15:00:00,Start
5,M1,2024-01-03 16:30:00,Stop
6,M2,2024-01-03 07:50:00,Start
7,M2,2024-01-03 09:10:00,Stop
8,M2,2024-01-03 10:30:00,Start
9,M2,2024-01-03 12:00:00,Stop


In [7]:
# move the new rows to the positions in the stop list
for my_index in reversed(stop_list):
    row_to_move = df.iloc[[len(df)-1]]  # Using double brackets keeps it as a DataFrame
    df_remaining = df.drop(df.index[len(df)-1])
    target_pos = my_index + 1
    df = pd.concat([df_remaining.iloc[:target_pos], row_to_move, df_remaining.iloc[target_pos:]])

In [8]:
df = df.reset_index(drop=True)
df['EventTime'] = pd.to_datetime(df['EventTime'])
df = df.sort_values(by=['Machine', 'EventType', 'EventTime'])
# This returns ONLY the 'Stop' rows, no NaNs created
df1 = df[df['EventType'] == 'Stop'].reset_index(drop=True)
df = df[df['EventType'] != 'Stop'].reset_index(drop=True)
df

,Machine,EventTime,EventType
0,M1,2024-01-03 08:00:00,Start
1,M1,2024-01-03 11:00:00,Start
2,M1,2024-01-03 15:00:00,Start
3,M2,2024-01-03 07:50:00,Start
4,M2,2024-01-03 10:30:00,Start
5,M2,2024-01-03 13:45:00,Start
6,M2,2024-01-03 17:10:00,Start
7,M3,2024-01-03 08:30:00,Start
8,M3,2024-01-03 10:10:00,Start
9,M3,2024-01-03 12:45:00,Start


In [9]:
df
#Merge 
# Joins df2 to df1 using their indices
# This adds '_left' and '_right' to the overlapping columns
df_merged = df.join(df1, lsuffix='_left', rsuffix='_right')

df_merged = df_merged.rename(columns={'Machine_left': 'Machine', 'EventTime_left':'StartTime', 'EventTime_right' : 'StopTime' })
df_merged = df_merged.drop(['Machine_right', 'EventType_left', 'EventType_right'], axis=1)
df_merged
df_merged['RunMinutes'] = (df_merged['StopTime'] - df_merged['StartTime']) /  pd.Timedelta(minutes=1)

df_merged

,Machine,StartTime,StopTime,RunMinutes
0,M1,2024-01-03 08:00:00,2024-01-03 10:15:00,135.0
1,M1,2024-01-03 11:00:00,2024-01-03 13:30:00,150.0
2,M1,2024-01-03 15:00:00,2024-01-03 16:30:00,90.0
3,M2,2024-01-03 07:50:00,2024-01-03 09:10:00,80.0
4,M2,2024-01-03 10:30:00,2024-01-03 12:00:00,90.0
5,M2,2024-01-03 13:45:00,2024-01-03 16:00:00,135.0
6,M2,2024-01-03 17:10:00,2024-01-03 18:00:00,50.0
7,M3,2024-01-03 08:30:00,2024-01-03 09:40:00,70.0
8,M3,2024-01-03 10:10:00,2024-01-03 11:30:00,80.0
9,M3,2024-01-03 12:45:00,2024-01-03 14:20:00,95.0
